In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Silver

In [0]:
log("Reading from Silver customers ...")
df_silver = spark.table(TBL_SILVER_CUSTOMERS)
df_silver.show()

## 2. Build dim_customer

In [0]:
df_dim_customer = df_silver.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("segment")
)

## 3. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_DIM_CUSTOMER):
    log(f"Upserting into {TBL_GOLD_DIM_CUSTOMER} ...")

    delta_table = DeltaTable.forName(spark, TBL_GOLD_DIM_CUSTOMER)
    (
        delta_table.alias("target")
        .merge(
            df_dim_customer.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log(f"✅ Done. Table {TBL_GOLD_DIM_CUSTOMER} upserted")

else:
    log(f"Creating {TBL_GOLD_DIM_CUSTOMER}")
    (
        df_dim_customer.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TBL_GOLD_DIM_CUSTOMER)
    )
    
    log(f"✅ Done. Table {TBL_GOLD_DIM_CUSTOMER} created")

In [0]:
display(spark.table(TBL_GOLD_DIM_CUSTOMER))